In [2]:
!pip install qiskit
!pip install qiskit matplotlib pylatexenc
!pip install qiskit_aer
!pip install jupyter
!pip install sympy
!pip install matplotlib
from qiskit.quantum_info import Statevector
from numpy import sqrt
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit.result import marginal_distribution
from qiskit.circuit.library import UGate
from math import pi
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 96.0 MB/s eta 0:00:00


In [3]:
!pip install qiskit-aer

***NOTE:***

Let's create a simple Simon oracle for:

s = 110

We will use 3 input qubits and 2 output qubits.

Define:

$$ f(x_0,x_1,x_2) = (x_0\oplus x_1,\;x_2) $$

Because:

$$ f(x)=f(x\oplus110) $$


In [4]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

n = 3
m = 2

oracle = QuantumCircuit(n + m)

# f(x) = (x0 XOR x1, x2)

# First output bit
oracle.cx(0, n)
oracle.cx(1, n)

# Second output bit
oracle.cx(2, n + 1)

print(oracle.draw())

                    
q_0: ──■────────────
       │            
q_1: ──┼─────────■──
       │         │  
q_2: ──┼────■────┼──
     ┌─┴─┐  │  ┌─┴─┐
q_3: ┤ X ├──┼──┤ X ├
     └───┘┌─┴─┐└───┘
q_4: ─────┤ X ├─────
          └───┘     


In [5]:
#Building Simon's Algorithm
qc = QuantumCircuit(n + m, n)

# Step 1: Create superposition on input register
qc.h(range(n))

# Step 2: Apply Simon oracle
qc.compose(oracle, inplace=True)

# Step 3: Hadamards again
qc.h(range(n))

# Step 4: Measure input register
qc.measure(range(n), range(n))

print(qc.draw())

     ┌───┐     ┌───┐          ┌─┐      
q_0: ┤ H ├──■──┤ H ├──────────┤M├──────
     ├───┤  │  └───┘     ┌───┐└╥┘   ┌─┐
q_1: ┤ H ├──┼─────────■──┤ H ├─╫────┤M├
     ├───┤  │         │  ├───┤ ║ ┌─┐└╥┘
q_2: ┤ H ├──┼────■────┼──┤ H ├─╫─┤M├─╫─
     └───┘┌─┴─┐  │  ┌─┴─┐└───┘ ║ └╥┘ ║ 
q_3: ─────┤ X ├──┼──┤ X ├──────╫──╫──╫─
          └───┘┌─┴─┐└───┘      ║  ║  ║ 
q_4: ──────────┤ X ├───────────╫──╫──╫─
               └───┘           ║  ║  ║ 
c: 3/══════════════════════════╩══╩══╩═
                               0  2  1 


In [14]:
#Running the simulator
simulator = AerSimulator()

compiled = transpile(qc, simulator)

result = simulator.run(
    compiled,
    shots=1024
).result()

counts = result.get_counts()

print(counts)

{'000': 257, '100': 266, '011': 251, '111': 250}


**The measurement distribution contains only strings orthogonal to the hidden secret.**

In [17]:
#Classical Post-Processing Part for solving the equations and finding teh hidden string
import numpy as np

# Output obtained from the Simon quantum circuit
counts = {
    '000': 267,
    '100': 226,
    '111': 261,
    '011': 270
}

# ---------------------------------------------------
# Step 1: Take the measured y values
# ---------------------------------------------------

y_values = list(counts.keys())

print("Measured y values:")
print(y_values)


# ---------------------------------------------------
# Step 2: Reverse Qiskit's bit order
# ---------------------------------------------------

# Qiskit displays classical bits from right to left.
# So reverse each string before doing the mathematics.

y_values = [y[::-1] for y in y_values]

print("\nY values used for equations:")
print(y_values)


# ---------------------------------------------------
# Step 3: Remove 000
# ---------------------------------------------------

# 000 gives the equation:
# 0 · s = 0
# which gives no information.

y_values = [y for y in y_values if y != "000"]

print("\nUseful equations:")
for y in y_values:
    print(y, "· s = 0 (mod 2)")


# ---------------------------------------------------
# Step 4: Brute-force all possible 3-bit strings
# ---------------------------------------------------

n = len(y_values[0])

solutions = []

for number in range(1, 2**n):

    # Convert number to binary string
    s = format(number, f"0{n}b")

    valid = True

    # Check every equation y · s = 0
    for y in y_values:

        dot_product = 0

        for yi, si in zip(y, s):

            # Multiplication followed by XOR
            dot_product ^= int(yi) & int(si)

        # Equation must equal 0
        if dot_product != 0:
            valid = False
            break

    if valid:
        solutions.append(s)


# ---------------------------------------------------
# Step 5: Display the answer
# ---------------------------------------------------

print("\nPossible non-zero solutions:")
print(solutions)

if len(solutions) == 1:
    print("\nHidden string =", solutions[0])
else:
    print("\nMore independent equations are required.")

Measured y values:
['000', '100', '111', '011']

Y values used for equations:
['000', '001', '111', '110']

Useful equations:
001 · s = 0 (mod 2)
111 · s = 0 (mod 2)
110 · s = 0 (mod 2)

Possible non-zero solutions:
['110']

Hidden string = 110
